In [1]:
import pandas as pd
import numpy as np

In [2]:
customer = pd.read_csv("D:\Project_Olist_clean\customer_cleaned.csv")
geolocation = pd.read_csv("D:\Project_Olist_clean\geolocation_cleaned.csv")
order_items= pd.read_csv("D:\Project_Olist_clean\order_items_cleaned.csv")
payments= pd.read_csv("D:\Project_Olist_clean\payments_cleaned.csv")
review=pd.read_csv("D:\\Project_Olist_clean\\review_cleaned.csv")
order=pd.read_csv("D:\Project_Olist_clean\order_cleaned.csv")
products=pd.read_csv("D:\Project_Olist_clean\products_cleaned.csv")
sellers=pd.read_csv("D:\Project_Olist_clean\sellers_cleaned.csv")
category=pd.read_csv("D:\Project_Olist_clean\category_cleaned.csv")

In [3]:
order["order_purchase_timestamp"] = pd.to_datetime(
    order["order_purchase_timestamp"],
    errors="coerce"
)

order["order_delivered_customer_date"] = pd.to_datetime(
    order["order_delivered_customer_date"],
    errors="coerce"
)

order["order_estimated_delivery_date"] = pd.to_datetime(
    order["order_estimated_delivery_date"],
    errors="coerce"
)

In [4]:
order_items["price"] = pd.to_numeric(
    order_items["price"],
    errors="coerce"
)

order_items["freight_value"] = pd.to_numeric(
    order_items["freight_value"],
    errors="coerce"
)

payments["payment_value"] = pd.to_numeric(
    payments["payment_value"],
    errors="coerce"
)

payments["payment_installments"] = pd.to_numeric(
    payments["payment_installments"],
    errors="coerce"
)

review["review_score"] = pd.to_numeric(
    review["review_score"],
    errors="coerce"
)


### Order merge customer

In [5]:
customer_order = order.merge(
    customer[
        [
            "customer_id",
            "customer_unique_id",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)
customer_order = customer_order[
    customer_order["order_status"] == "delivered"
].copy()

### Xác định customer có first order

In [6]:
customer_order = customer_order.sort_values(
    [
        "customer_unique_id",
        "order_purchase_timestamp",
        "order_id"
    ]
)

first_orders = (
    customer_order
    .drop_duplicates(
        subset="customer_unique_id",
        keep="first"
    )
    .copy()
)

print("Số customer có first order:", len(first_orders))


Số customer có first order: 93336


### First order value

In [7]:
order_items["item_total_value"] = (
    order_items["price"]
    + order_items["freight_value"]
)

order_value = (
    order_items
    .groupby("order_id", as_index=False)
    ["item_total_value"]
    .sum()
    .rename(
        columns={
            "item_total_value": "first_order_value"
        }
    )
)


first_orders = first_orders.merge(
    order_value,
    on="order_id",
    how="left"
)

### First category

In [8]:
first_item = (
    order_items
    .sort_values(
        ["order_id", "order_item_id"]
    )
    .drop_duplicates(
        subset="order_id",
        keep="first"
    )
    [
        [
            "order_id",
            "product_id"
        ]
    ]
)


first_item = first_item.merge(
    products[
        [
            "product_id",
            "product_category_name",
            "product_category_name_english"
        ]
    ],
    on="product_id",
    how="left"
)


first_item["first_category"] = (
    first_item["product_category_name_english"]
    .fillna(
        first_item["product_category_name"]
    )
    .fillna("unknown")
)


first_category = first_item[
    [
        "order_id",
        "first_category"
    ]
]


first_orders = first_orders.merge(
    first_category,
    on="order_id",
    how="left"
)


### First review score

In [14]:
if "review_creation_date" in review.columns:

    review["review_creation_date"] = pd.to_datetime(
        review["review_creation_date"],
        errors="coerce"
    )

    first_review = (
        review
        .sort_values(
            [
                "order_id",
                "review_creation_date",
                "review_id"
            ]
        )
        .drop_duplicates(
            subset="order_id",
            keep="first"
        )
        [
            [
                "order_id",
                "review_score"
            ]
        ]
    )

else:

    first_review = (
        review[
            [
                "order_id",
                "review_score"
            ]
        ]
        .drop_duplicates(
            subset="order_id",
            keep="first"
        )
    )


first_orders = first_orders.merge(
    first_review,
    on="order_id",
    how="left"
)

first_orders.rename(
    columns={
        "review_score": "first_review_score"
    },
    inplace=True
)


### First delivery days

In [9]:
first_orders["first_delivery_days"] = (
    first_orders["order_delivered_customer_date"]
    - first_orders["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

first_orders["first_delivery_days"] = (
    first_orders["first_delivery_days"]
    .round(2)
)

### First late order (0,1)

In [10]:
first_orders["first_late_order"] = np.where(
    first_orders["order_delivered_customer_date"]
    >
    first_orders["order_estimated_delivery_date"],
    1,
    0
)

# Nếu thiếu một trong hai ngày thì không xác định được late.
first_orders.loc[
    first_orders[
        [
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ].isna().any(axis=1),
    "first_late_order"
] = np.nan

### Payments

In [11]:
first_payment = (
    payments
    .sort_values(
        [
            "order_id",
            "payment_value",
            "payment_sequential"
        ],
        ascending=[True, False, True]
    )
    .drop_duplicates(
        subset="order_id",
        keep="first"
    )
    [
        [
            "order_id",
            "payment_type",
            "payment_installments"
        ]
    ]
)


first_orders = first_orders.merge(
    first_payment,
    on="order_id",
    how="left"
)


first_orders.rename(
    columns={
        "payment_type": "first_payment_type",
        "payment_installments": "first_payment_installments"
    },
    inplace=True
)


### Tạo cột Target

In [12]:
dataset_end_date = customer_order[
    "order_purchase_timestamp"
].max()

print(
    "Dataset end date:",
    dataset_end_date
)


# ------------------------------------------------------------
# Chỉ giữ first order có đủ 90 ngày follow-up
# ------------------------------------------------------------

first_orders = first_orders[
    first_orders["order_purchase_timestamp"]
    <= dataset_end_date - pd.Timedelta(days=90)
].copy()


print(
    "Customer đủ 90 ngày:",
    len(first_orders)
)


# ------------------------------------------------------------
# Tạo bảng order history
# ------------------------------------------------------------

order_history = customer_order[
    [
        "customer_unique_id",
        "order_id",
        "order_purchase_timestamp"
    ]
].copy()


# ------------------------------------------------------------
# Join first order với order history
# ------------------------------------------------------------

first_for_target = first_orders[
    [
        "customer_unique_id",
        "order_id",
        "order_purchase_timestamp"
    ]
].rename(
    columns={
        "order_id": "first_order_id",
        "order_purchase_timestamp": "first_order_date"
    }
)


target_check = first_for_target.merge(
    order_history,
    on="customer_unique_id",
    how="left"
)


# ------------------------------------------------------------
# Số ngày từ first order đến order tiếp theo
# ------------------------------------------------------------

target_check["days_after_first"] = (
    target_check["order_purchase_timestamp"]
    - target_check["first_order_date"]
).dt.total_seconds() / (24 * 60 * 60)


# ------------------------------------------------------------
# Xác định repeat trong 90 ngày
# ------------------------------------------------------------

repeat_condition = (
    (target_check["order_id"] != target_check["first_order_id"])
    &
    (target_check["days_after_first"] > 0)
    &
    (target_check["days_after_first"] <= 90)
)


repeat_customers = (
    target_check.loc[
        repeat_condition,
        "customer_unique_id"
    ]
    .unique()
)




first_orders["repeat_90d"] = (
    first_orders["customer_unique_id"]
    .isin(repeat_customers)
    .astype(int)
)

Dataset end date: 2018-08-29 15:00:37
Customer đủ 90 ngày: 75304


In [15]:
customer_ml = first_orders[
    [
        "customer_unique_id",

        "first_order_value",

        "first_category",

        "first_review_score",

        "first_delivery_days",

        "first_late_order",

        "customer_state",

        "first_payment_type",

        "first_payment_installments",

        "repeat_90d"
    ]
].copy()


### Xử lý missing

In [23]:
# Categorical
for col in [
    "first_category",
    "customer_state",
    "first_payment_type"
]:
    customer_ml[col] = (
        customer_ml[col]
        .fillna("unknown")
        .astype(str)
    )


# Numerical
customer_ml["first_order_value"] = pd.to_numeric(
    customer_ml["first_order_value"],
    errors="coerce"
)

customer_ml["first_review_score"] = pd.to_numeric(
    customer_ml["first_review_score"],
    errors="coerce"
)

customer_ml["first_delivery_days"] = pd.to_numeric(
    customer_ml["first_delivery_days"],
    errors="coerce"
)

customer_ml["first_payment_installments"] = pd.to_numeric(
    customer_ml["first_payment_installments"],
    errors="coerce"
)


# Median imputation
customer_ml["first_review_score"] = (
    customer_ml["first_review_score"]
    .fillna(
        customer_ml["first_review_score"].median()
    )
)

customer_ml["first_delivery_days"] = (
    customer_ml["first_delivery_days"]
    .fillna(
        customer_ml["first_delivery_days"].median()
    )
)

customer_ml["payment_installments"] = (
    customer_ml["payment_installments"]
    .fillna(
        customer_ml["payment_installments"].median()
    )
)


# Late order
customer_ml["first_late_order"] = (
    customer_ml["first_late_order"]
    .fillna(0)
    .astype(int)
)



### Loại những row ko có order

In [24]:
customer_ml = customer_ml.dropna(
    subset=["first_order_value"]
)

In [25]:
duplicate_count = (
    customer_ml["customer_unique_id"]
    .duplicated()
    .sum()
)

print(
    "Duplicate customer_unique_id:",
    duplicate_count
)

print(
    customer_ml["repeat_90d"]
    .value_counts()
)


print(
    customer_ml["repeat_90d"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)




print("\n===== MISSING =====")

print(
    customer_ml.isna().sum()
)


# ============================================================
# 21. KIỂM TRA FINAL TABLE
# ============================================================

print("\n===== SHAPE =====")

print(
    customer_ml.shape
)

print("\n===== COLUMNS =====")

print(
    customer_ml.columns.tolist()
)

print("\n===== SAMPLE =====")

print(
    customer_ml.head()
)




Duplicate customer_unique_id: 0
repeat_90d
0    73790
1     1514
Name: count, dtype: int64
repeat_90d
0    97.99
1     2.01
Name: proportion, dtype: float64

===== MISSING =====
customer_unique_id            0
first_order_value             0
first_category                0
first_review_score            0
first_delivery_days           0
first_late_order              0
customer_state                0
first_payment_type            0
first_payment_installments    1
repeat_90d                    0
payment_installments          0
dtype: int64

===== SHAPE =====
(75304, 11)

===== COLUMNS =====
['customer_unique_id', 'first_order_value', 'first_category', 'first_review_score', 'first_delivery_days', 'first_late_order', 'customer_state', 'first_payment_type', 'first_payment_installments', 'repeat_90d', 'payment_installments']

===== SAMPLE =====
                 customer_unique_id  first_order_value  first_category  \
0  0000366f3b9a7992bf8c76cfdf3221e2             141.90  bed_bath_table   
1 

In [26]:
customer_ml.to_csv(
    r"D:\Project_Olis\cleaned\customer_ml.csv",
    index=False
)